In [1]:
# pip install sentence-transformers
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np, time

bi = SentenceTransformer("all-MiniLM-L6-v2")          # stage 1: retriever (bi-encoder)
ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")  # stage 2: reranker (cross-encoder)

c:\Users\durga\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
corpus = [
     "To reset a franchise admin password, open the admin console and choose account recovery.",# 0
     "Password requirements: minimum twelve characters with one symbol and one number.",# 1       # lexical trap for 'password'
     "Never share your admin password with staff; each user gets their own login.",# 2             # lexical trap
     "Refunds are processed within seven business days to the original payment method.",# 3
     "Refund requests must be submitted within thirty days of the original purchase.",# 4           # partial-answer trap
     "Reimbursement for approved franchise expenses appears on your next monthly statement.",# 5
     "Error code E-4021 indicates a payment gateway timeout during checkout.",# 6
     "If checkout fails, verify the customer's card details and retry the transaction.",# 7          # lexical trap for 'checkout'
     "Monthly royalty fees are calculated as a percentage of gross sales.",# 8
     "Royalty payments are due by the fifth of each month via the financial portal.",# 9
     "The onboarding wizard guides new franchisees through initial store setup.",# 10
     "New franchisee training covers POS operation, inventory, and staff scheduling.",# 11
     "The point of sale terminal syncs inventory every fifteen minutes.",# 12
     "Inventory discrepancies should be reported to your regional manager within 24 hours.",# 13
     "Marketing materials can be requested through the brand asset portal.",# 14
     "Seasonal promotion templates are available in the marketing resource library.",# 15
     "Store hours can be updated under settings, location, and operating schedule.",# 16
     "To close your store temporarily, submit a closure request to head office.",# 17
     "Employee shift schedules are managed in the workforce planning dashboard.",# 18
     "Payroll runs biweekly; timesheets must be approved by end of day Friday.",# 19
]

queries = [
    "how do I reset my admin password",
    "how long do refunds take",
    "what does error E-4021 mean",
    "how are royalty fees calculated",
    "when are royalty payments due",
    "how do new franchisees get trained",
    "how often does inventory sync",
    "where do I get marketing materials",
    "how do I change my store hours",
    "how do I temporarily close my store",
    "how do I get reimbursed for expenses",
    "what are the password requirements",
    "who do I report inventory problems to",
    "how do I manage employee shifts",
    "when does payroll run",
    "how do I recover a locked account",
    "can I get my money back after a purchase",   # paraphrase — dense should win retrieval
    "what happens if checkout fails",
    "where are seasonal promotion templates",
    "how do I approve timesheets",
]

In [3]:
import random; random.seed(0)
vocab = "staff schedule shift report dashboard notification setting profile district audit compliance training module ticket store region".split()
filler = [" ".join(random.choices(vocab, k=random.randint(10, 30))) for _ in range(180)]
corpus_full = corpus + filler   # 20 planted + 180 filler = 200
doc_emb = bi.encode(corpus_full, normalize_embeddings=True)

In [4]:
def retrieve(query, k=30):
    q = bi.encode([query], normalize_embeddings=True)[0]
    sims = doc_emb @ q
    idx = np.argsort(-sims)[:k]
    return [(int(i), float(sims[i])) for i in idx]

cands = retrieve(queries[0], k=30)
print(cands[:5])   # bi-encoder's own top-5, before reranking

[(0, 0.682278573513031), (2, 0.441694051027298), (1, 0.35625359416007996), (17, 0.19781024754047394), (87, 0.17779642343521118)]


In [5]:
def rerank(query, candidate_indices):
    pairs = [[query, corpus_full[i]] for i in candidate_indices]   # query paired with EACH doc
    scores = ce.predict(pairs)                                     # one score per pair
    reranked = sorted(zip(candidate_indices, scores), key=lambda x: -x[1])
    return [(int(i), float(s)) for i, s in reranked]

cand_idx = [i for i, _ in cands]
reranked = rerank(queries[0], cand_idx)
print(reranked[:5])   # cross-encoder's top-5, after reranking

[(0, 7.441977024078369), (2, -0.9151705503463745), (1, -9.861069679260254), (17, -11.158441543579102), (75, -11.250862121582031)]


In [6]:
def timed_pipeline(query, k=30):
    t0 = time.perf_counter()
    cands = retrieve(query, k=k)
    t1 = time.perf_counter()
    reranked = rerank(query, [i for i, _ in cands])
    t2 = time.perf_counter()
    return (t1 - t0) * 1000, (t2 - t1) * 1000   # retrieve_ms, rerank_ms

r_ms, rr_ms = [], []
for q in queries:
    a, b = timed_pipeline(q)
    r_ms.append(a); rr_ms.append(b)

print(f"retrieve: {np.mean(r_ms):6.1f} ms avg")
print(f"rerank  : {np.mean(rr_ms):6.1f} ms avg   ({np.mean(rr_ms)/np.mean(r_ms):.0f}x slower)")

retrieve:   16.1 ms avg
rerank  :  185.9 ms avg   (12x slower)


In [7]:
for q in queries:
    cands = retrieve(q, k=30)
    before = [i for i, _ in cands][:5]
    after  = [i for i, _ in rerank(q, [i for i, _ in cands])][:5]
    moved  = len(set(before) ^ set(after)) // 2   # how many docs swapped in/out of top-5
    # did the reranker pull something from deep in the candidate list up to #1?
    rank_of_new_top1 = [i for i, _ in cands].index(after[0])
    print(f"{q[:38]:40} top5 changed:{moved}  new #1 came from bi-rank {rank_of_new_top1}")

how do I reset my admin password         top5 changed:1  new #1 came from bi-rank 0
how long do refunds take                 top5 changed:1  new #1 came from bi-rank 0
what does error E-4021 mean              top5 changed:1  new #1 came from bi-rank 0
how are royalty fees calculated          top5 changed:1  new #1 came from bi-rank 0
when are royalty payments due            top5 changed:0  new #1 came from bi-rank 0
how do new franchisees get trained       top5 changed:1  new #1 came from bi-rank 0
how often does inventory sync            top5 changed:2  new #1 came from bi-rank 0
where do I get marketing materials       top5 changed:3  new #1 came from bi-rank 0
how do I change my store hours           top5 changed:3  new #1 came from bi-rank 0
how do I temporarily close my store      top5 changed:3  new #1 came from bi-rank 0
how do I get reimbursed for expenses     top5 changed:0  new #1 came from bi-rank 0
what are the password requirements       top5 changed:2  new #1 came from bi

Re ranker- An encoder that has been trained to take both doc and query at the same and output the relevancy score for the doc and query. A cross encoder to be specific. There are 2 types of encoders. Bi and cross. Bi means two passes. Using BI encoder we will first convert the entire corpus of millions to vectors once and store in a vector database . When user query comes, the same bi encoder converts the query to vector and extracts the relevant docs from millions of docs using cosine similarity. As vectors are pre-computed it won't take that long and is practically fast. From the millions of docs it extracts 30 relevant docs so its recall is high, but the ranknig of docs in that final 30 short list is often not great, as bi encoder retrieves based on conceptual relevance, it doesn't rank the docs in the short list based on whether they actually answer the query at hand. To effecitvely handle this the cross encoder takes both query and each doc from the short list and then scores them with the idea, how much this doc would be useful to answer the query. We then re arrange the short list with the updated re ranker scores and often take top 5 from the list. At rerank time the model runs a full forward pass on each (query, doc) pair — 30 passes per query — which is why it costs more.. From our lab we found the metrics as retrieve:   16.1 ms avg , rerank  :  185.9 ms avg   (12x slower). So for each query to retrieve the 30 docs from full corpus took less time than re ranking the docs from their output. As we can see currently we are on cpu , that's why the difference is clear with 12x slower. We can also see that after reranking approx 2-3 docs have changed their positions with respect to when they are retrieved using bi encoder. As this is small dataset its not that clear. On this small clean corpus only 2–3 docs shifted in the top-5. In production the bi-encoder's top-30 is packed with near-duplicate, topically-similar-but-wrong passages, so the cross-encoder reorders far more aggressively — that's where its value shows up.

bi-encoder = high recall, weak ranking. Cross-encoder = fixes the ranking on the shortlist. Reranking improves precision

Reranker — a cross-encoder trained to take a query and a document together and output a single relevancy score for that pair. There are two encoder types: bi and cross. A bi-encoder makes two separate passes — we encode the entire corpus of millions of docs into vectors once, offline, and store them in a vector DB; when a query arrives, the same bi-encoder encodes just the query and pulls the closest docs by cosine similarity. Because the doc vectors are precomputed, this is fast. Run wide (top-30), its recall is high — the right doc is almost always somewhere in those 30 — but its ranking within that shortlist is weak, because it matches on conceptual/topical relevance and never judges whether a doc actually answers the specific query. The cross-encoder fixes this: it takes the query and each shortlisted doc together and scores how well that doc answers the query. We re-sort the 30 by these scores and usually keep the top 5. It's slower because it can't precompute anything — the score depends on the live query, so each (query, doc) pair needs its own full forward pass, i.e. 30 passes per query.

Measured in the lab: retrieve 16.1 ms avg, rerank 185.9 ms avg — reranking was the expensive stage at ~12× the retrieval cost, exactly because of those 30 per-query forward passes. We're on CPU here, so the gap is exaggerated; on a GPU the rerank cost drops sharply, but the ordering (rerank ≫ retrieve) holds. On movement: only ~2–3 docs shifted in the top-5 after reranking, because this is a small, clean corpus with few real distractors. In production the bi-encoder's top-30 is packed with near-duplicate, topically-similar-but-wrong passages, so the cross-encoder reorders far more aggressively — and that's where its value shows up. 

Net: bi-encoder = cheap, high recall, weak ranking, precomputable → stage 1 retriever over the full corpus. Cross-encoder = expensive, query-specific relevance, not precomputable → stage 2 reranker on the shortlist. Retrieve wide and cheap, rerank narrow and accurate.